# 14 Case Study — Solution

The complete solution for the Songbai Nursing Home Legionnaires' disease mini outbreak investigation report.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || True
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy import stats

# -- CJK font setup (prevents CJK labels from rendering as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150


## Question 1: Outbreak summary table

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

n_total = len(df)
n_infected = int(df["infected"].sum())
n_deaths = int((df["outcome"] == "dead").sum())
n_hosp = int(df["hospitalized"].sum())
n_icu = int(df["icu_admission"].sum())

summary = pd.DataFrame([
    ["Total residents", n_total, ""],
    ["Infected", n_infected, f"{n_infected/n_total:.1%}"],
    ["Deaths", n_deaths, f"{n_deaths/n_infected:.1%} (CFR)"],
    ["Hospitalized", n_hosp, f"{n_hosp/n_infected:.1%} (hosp. rate)"],
    ["ICU", n_icu, f"{n_icu/n_hosp:.1%} (ICU/hosp.)"],
    ["Attack rate", f"{n_infected/n_total:.1%}", ""],
    ["Case fatality rate", f"{n_deaths/n_infected:.1%}", ""],
], columns=["Metric", "Value", "Proportion"])

print("=== Outbreak summary table ===")
print(summary.to_string(index=False))

## Question 2: Quick risk-factor screening

In [ ]:
factors = ["shower_use", "hydrotherapy_use", "comorbidity_copd", "immunosuppressed"]
results = []

for factor in factors:
    exposed_inf = int(df[(df[factor] == 1) & (df["infected"] == 1)].shape[0])
    exposed_n = int(df[df[factor] == 1].shape[0])
    unexposed_inf = int(df[(df[factor] == 0) & (df["infected"] == 1)].shape[0])
    unexposed_n = int(df[df[factor] == 0].shape[0])

    ar_exp = exposed_inf / exposed_n if exposed_n > 0 else 0
    ar_unexp = unexposed_inf / unexposed_n if unexposed_n > 0 else 0
    rr = ar_exp / ar_unexp if ar_unexp > 0 else float("inf")

    chi2, p, _, _ = stats.chi2_contingency(
        pd.crosstab(df[factor], df["infected"])
    )

    results.append({
        "factor": factor,
        "exposed_AR": f"{ar_exp:.1%}",
        "unexposed_AR": f"{ar_unexp:.1%}",
        "RR": f"{rr:.2f}",
        "p-value": f"{p:.4f}",
        "sig": "*" if p < 0.05 else "",
    })

rr_df = pd.DataFrame(results)
print("=== Risk-factor RR comparison table ===")
print(rr_df.to_string(index=False))

max_rr = rr_df.loc[rr_df["RR"].astype(float).idxmax()]
print(f"\n→ Factor with the largest RR: {max_rr['factor']} (RR = {max_rr['RR']})")
print("→ Shower use is the strongest exposure risk factor and is significantly associated with infection status")

## Question 3 (Challenge): Mini SitRep

In [ ]:
cases = df[df["infected"] == 1].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Chart 1: Epidemic curve ---
daily = cases.groupby("symptom_onset_date").size()
full_range = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
daily = daily.reindex(full_range, fill_value=0)

axes[0].bar(daily.index, daily.values, color="steelblue", edgecolor="white")
peak = daily.idxmax()
axes[0].axvline(peak, color="red", linestyle="--", alpha=0.7)
axes[0].set_title(f"Epidemic curve (peak: {peak.strftime('%m/%d')})")
axes[0].set_ylabel("Daily new cases")
axes[0].tick_params(axis="x", rotation=45)

# --- Chart 2: Age distribution ---
for label, grp in df.groupby("infected"):
    tag = "Infected" if label == 1 else "Not infected"
    axes[1].hist(grp["age"], bins=15, alpha=0.6, label=tag, edgecolor="white")
axes[1].set_title("Age distribution")
axes[1].set_xlabel("Age")
axes[1].legend()

# --- Chart 3: Attack rate by floor and wing ---
zone = df.groupby(["floor", "wing"])["infected"].agg(["sum", "count"]).reset_index()
zone["ar"] = zone["sum"] / zone["count"] * 100
zone["label"] = zone["floor"].astype(str) + "F-" + zone["wing"]
colors = ["#e74c3c" if ar > 50 else "steelblue" for ar in zone["ar"]]
axes[2].bar(zone["label"], zone["ar"], color=colors)
axes[2].axhline(50, color="red", linestyle="--", alpha=0.5)
axes[2].set_title("Attack Rate by Floor and Wing")
axes[2].set_ylabel("Attack Rate (%)")
for i, row in zone.iterrows():
    axes[2].text(i, row["ar"] + 1, f"{row['ar']:.0f}%", ha="center", fontsize=9)

plt.suptitle("Songbai Nursing Home Legionnaires' Disease Outbreak — Mini SitRep", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Action recommendations
top_zone = zone.sort_values("ar", ascending=False).iloc[0]
print("=" * 40)
print("  Action recommendations")
print("=" * 40)
print(f"  1. Priority zone to address: {top_zone['label']} (attack rate {top_zone['ar']:.1f}%)")
print(f"  2. Secondary focus: 2F-A (54.5%) — these two zones account for the majority of cases")
print(f"  3. Immediately shut down showers in the high-risk zones")
print(f"  4. Conduct environmental sampling of the 2F and 3F plumbing systems")
print("=" * 40)

### Interpretation

- **Question 1**: The summary table is the first page of an outbreak report, letting decision-makers quickly grasp the scale
- **Question 2**: RR screening quickly identifies the exposure factors most worth investigating in depth
  - Note: the crude RR is not adjusted for confounders; pair it with stratified analysis (Ch05) and logistic regression (Ch06)
- **Question 3**: A good SitRep must include "action recommendations" -- the purpose of analysis is to support decisions

Congratulations on finishing the final exercise! You now have the core skills to conduct an outbreak investigation with Python.

## Question 4: Norovirus banquet cluster mini outbreak investigation (Norovirus scenario)

A gastroenteritis cluster breaks out after a banquet. Conduct a complete mini outbreak investigation from the guests' food exposure and onset line list.

1. Plot the epidemic curve (onset time) and determine the transmission pattern
2. Calculate the attack rate and risk ratio (RR) for each food item
3. Identify the suspect food with the highest RR and run a chi-square test
4. Write a conclusion: the suspected source of infection and what the epidemic curve shape implies

In [ ]:
# Norovirus banquet cluster: food exposure and onset line list for 150 guests
from epi_learning.metrics import attack_rate, risk_ratio
rng = np.random.default_rng(1404)
n = 150
foods = ["生蠔", "沙拉", "甜點", "湯品"]
ate = {f: rng.binomial(1, 0.5, n) for f in foods}
p_ill = (0.05 + 0.7 * ate["生蠔"]).clip(0, 1)     # oysters are contaminated
ill = rng.binomial(1, p_ill)
onset_hr = np.where(ill == 1, rng.normal(32, 8, n).clip(6, 72), np.nan)  # norovirus incubation ~24-48h
guests = pd.DataFrame({"guest_id": range(1, n + 1), "ill": ill,
                       **{f: ate[f] for f in foods}, "onset_hr": np.round(onset_hr, 0)})
print(f"Banquet: {n} guests, {ill.sum()} ill ({ill.mean():.1%})")

sick = guests[guests["ill"] == 1]
fig, ax = plt.subplots(figsize=(7, 3.5))
bins = range(0, 78, 6)
ax.hist(sick["onset_hr"], bins=bins, color="#D97757", edgecolor="white")
ax.set_xlabel("Onset time (hours after banquet)"); ax.set_ylabel("Number of cases")
ax.set_title("Norovirus Banquet Cluster Epidemic Curve"); plt.tight_layout(); plt.show()

print("Risk ratio by food item:")
results = []
for f in foods:
    a = guests[(guests[f] == 1) & (guests.ill == 1)].shape[0]
    b = guests[(guests[f] == 1) & (guests.ill == 0)].shape[0]
    c = guests[(guests[f] == 0) & (guests.ill == 1)].shape[0]
    d = guests[(guests[f] == 0) & (guests.ill == 0)].shape[0]
    ar_e = attack_rate(a, a + b); ar_u = attack_rate(c, c + d)
    rr = risk_ratio(a, a + b, c, c + d)
    chi2, p, _, _ = stats.chi2_contingency([[a, b], [c, d]])
    results.append((f, ar_e, ar_u, rr, p))
    print(f"  {f}: ate={ar_e:.1%} did not eat={ar_u:.1%} RR={rr:.2f} p={p:.3g}")

culprit = max(results, key=lambda r: r[3])
print(f"\n[Conclusion] Most likely source of infection: {culprit[0]} (RR={culprit[3]:.2f}, p={culprit[4]:.3g}).")
print("The epidemic curve is unimodal with a clustered incubation period -> consistent with a point-source exposure, matching a one-time contamination of the food.")

## Question 5: COVID-19 workplace cluster investigation (COVID-19 scenario)

A company has a COVID-19 cluster, and a company-wide meeting is suspected to be the exposure event.

1. Plot the epidemic curve (by onset day)
2. Calculate the attack rate by department and identify the highest-risk department
3. Calculate the risk ratio (RR) for "attended the meeting"
4. Write a conclusion: was the meeting a suspected exposure?

In [ ]:
# COVID-19 workplace cluster: 200 employees at a company, a company-wide meeting as the suspected exposure
from epi_learning.metrics import risk_ratio
rng = np.random.default_rng(1405)
n = 200
dept = rng.choice(["業務", "研發", "行政", "客服"], n, p=[0.3, 0.3, 0.2, 0.2])
meeting = rng.binomial(1, np.where(dept == "業務", 0.9, 0.4))   # sales staff mostly attended
p_inf = (0.03 + 0.35 * meeting).clip(0, 1)
infected = rng.binomial(1, p_inf)
onset_day = np.where(infected == 1, rng.integers(2, 10, n), -1)   # days after the meeting when symptoms started
staff = pd.DataFrame({"emp_id": range(1, n + 1), "dept": dept,
                      "meeting": meeting, "infected": infected, "onset_day": onset_day})
print(f"Company: {n} employees, {infected.sum()} confirmed cases; {meeting.sum()} attended the meeting")

curve = staff[staff.infected == 1].groupby("onset_day").size()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(curve.index, curve.values, color="#D97757", edgecolor="white")
ax.set_xlabel("Days after the meeting"); ax.set_ylabel("Confirmed cases")
ax.set_title("COVID-19 Workplace Cluster Epidemic Curve"); plt.tight_layout(); plt.show()

by_dept = staff.groupby("dept").agg(n=("emp_id", "size"), cases=("infected", "sum"))
by_dept["attack_rate"] = (by_dept["cases"] / by_dept["n"]).round(3)
print(by_dept.sort_values("attack_rate", ascending=False).to_string())

a = staff[(staff.meeting == 1) & (staff.infected == 1)].shape[0]
b = staff[(staff.meeting == 1) & (staff.infected == 0)].shape[0]
c = staff[(staff.meeting == 0) & (staff.infected == 1)].shape[0]
d = staff[(staff.meeting == 0) & (staff.infected == 0)].shape[0]
rr = risk_ratio(a, a + b, c, c + d)
print(f"\nRR for attending the meeting = {rr:.2f}")
print(f"[Conclusion] The meeting is a suspected exposure event (RR={rr:.2f}); the sales department had the highest attendance rate, hence the highest attack rate,")
print("Recommendation: trace meeting contacts, improve ventilation, and increase screening.")

## Question 6 (Challenge): Dengue fever community outbreak SitRep (Dengue fever scenario)

A community is experiencing a dengue fever epidemic, and you need to produce a situation report (SitRep).

1. Plot the community-wide weekly epidemic curve
2. Calculate the cumulative incidence rate per 100,000 population for each district and rank the hotspots
3. Determine the epidemic trend (rising / stable / declining)
4. Write a 3-5 sentence SitRep: scale, hotspots, trend, and prevention recommendations

In [ ]:
# Dengue fever community outbreak: weekly case counts and population for 5 districts over 10 weeks (Challenge: write a SitRep)
rng = np.random.default_rng(1406)
districts = ["安南區", "三民區", "北屯區", "板橋區", "中西區"]
pop = {"安南區": 190000, "三民區": 340000, "北屯區": 280000, "板橋區": 550000, "中西區": 78000}
weekly_rate = {"安南區": 3.0, "三民區": 1.2, "北屯區": 0.8, "板橋區": 0.6, "中西區": 1.4}  # per 100k/week
_rows = []
for wk in range(1, 11):
    growth = 1.0 + 0.15 * wk    # epidemic growing week over week
    for d in districts:
        cases = rng.poisson(weekly_rate[d] * growth * pop[d] / 100000)
        _rows.append({"epi_week": wk, "district": d, "cases": cases})
dengue = pd.DataFrame(_rows)
region_pop = pd.DataFrame({"district": districts, "population": [pop[d] for d in districts]})
print(f"Dengue fever: {dengue['cases'].sum()} cases, 10 weeks x {len(districts)} districts")

weekly = dengue.groupby("epi_week")["cases"].sum()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(weekly.index, weekly.values, color="#D97757", edgecolor="white")
ax.set_xlabel("Epi week"); ax.set_ylabel("Weekly case count")
ax.set_title("Dengue Fever Community Weekly Epidemic Curve"); plt.tight_layout(); plt.show()

by_dist = dengue.groupby("district")["cases"].sum().reset_index()
by_dist = by_dist.merge(region_pop, on="district")
by_dist["rate_per_100k"] = (by_dist["cases"] / by_dist["population"] * 100000).round(1)
by_dist = by_dist.sort_values("rate_per_100k", ascending=False)
print(by_dist.to_string(index=False))

first_half = weekly.iloc[:5].sum(); second_half = weekly.iloc[5:].sum()
trend = "rising" if second_half > first_half * 1.1 else ("declining" if second_half < first_half * 0.9 else "stable")
top = by_dist.iloc[0]
print(f"\n[SitRep] This community has had {dengue['cases'].sum()} cumulative dengue fever cases over 10 weeks.")
print(f"The hotspot is {top['district']} ({top['rate_per_100k']}/100k, the highest of all districts).")
print(f"Weekly cases show a {trend} trend ({second_half} cases in the last 5 weeks vs {first_half} in the first 5 weeks).")
print("Recommendation: intensify breeding-site removal and spraying in the hotspot, and expand public health education and medical reporting.")